# SECCIÓN 2
## Preparación y validación de los datos

Esta sección continúa el trabajo de exploración inicial. No vuelve a cargar el dataset ni repite los controles generales ya realizados. Su objetivo es revisar la unidad de análisis, justificar las transformaciones necesarias y dejar una tabla con un registro por pedido para el análisis posterior.

## Etapa 0 — Punto de partida y decisiones pendientes

La exploración inicial registró 180.519 filas y 53 columnas, sin filas completamente duplicadas. También identificó valores faltantes en columnas que no son necesarias para el análisis principal, como `Product Description` y `Order Zipcode`.

Antes de comparar ganancias, necesitamos tener en cuenta que la base está organizada por artículos de pedido: un mismo `Order Id` puede aparecer en varias filas. Por eso, la tabla original no debe tratarse directamente como si cada fila fuera un pedido.

La pregunta que viene trabajando el grupo se refiere a las diferencias en la ganancia promedio por pedido según el estado de entrega y a cómo varían por segmento de clientes y región. Para responderla, primero debemos establecer cómo resumir las filas de cada pedido. La interpretación final debe basarse en los resultados del análisis del grupo.

## Etapa 1 — Validar la unidad de análisis y las variables económicas

### 1.1. Comprobar cuántas filas tiene cada pedido

Contamos las filas asociadas a cada `Order Id`. Esto permite confirmar si la unidad de observación original coincide con la unidad que necesita nuestra pregunta de investigación.

In [4]:
filas_por_pedido = df.groupby("Order Id").size()

print(f"Filas de la base original: {len(df):,}")
print(f"Pedidos únicos: {df['Order Id'].nunique():,}")
print(f"Pedidos con una sola fila: {(filas_por_pedido == 1).sum():,}")
print(f"Pedidos con varias filas: {(filas_por_pedido > 1).sum():,}")
print(f"Máximo de filas para un pedido: {filas_por_pedido.max()}")

Filas de la base original: 180,519
Pedidos únicos: 65,752
Pedidos con una sola fila: 19,850
Pedidos con varias filas: 45,902
Máximo de filas para un pedido: 5


### 1.2. Revisar un pedido con varias filas

Observamos un ejemplo con varias filas para comprobar que los registros se distinguen por `Order Item Id` y que los valores económicos pueden cambiar entre artículos. Esto ayuda a evaluar si corresponde resumir las ganancias a nivel de pedido.

In [5]:
id_ejemplo = filas_por_pedido[filas_por_pedido > 1].index[0]

columnas_ejemplo = [
    "Order Id",
    "Order Item Id",
    "Order Item Quantity",
    "Order Item Total",
    "Order Item Profit Ratio",
    "Order Profit Per Order",
    "Benefit per order",
    "Delivery Status"
]

display(
    df.loc[df["Order Id"] == id_ejemplo, columnas_ejemplo]
    .sort_values("Order Item Id")
)

,Order Id,Order Item Id,Order Item Quantity,Order Item Total,Order Item Profit Ratio,Order Profit Per Order,Benefit per order,Delivery Status
77011,2,2,1,193.99,0.47,91.18,91.18,Advance shipping
109322,2,3,5,227.50,0.30,68.25,68.25,Advance shipping
87884,2,4,1,107.89,0.34,36.47,36.47,Advance shipping


### 1.3. Comprobar qué variables se mantienen constantes dentro de cada pedido

El diccionario describe `Benefit per order` como ganancias por pedido. Sin embargo, en la base, `Benefit per order` y `Order Profit Per Order` coinciden fila por fila, pero sus valores pueden variar entre las filas asociadas a un mismo pedido. Por eso, el nombre de las variables por sí solo no alcanza para determinar cómo debe calcularse la ganancia total del pedido.

Comprobamos también que el riesgo, el estado de entrega, el segmento y la región se mantengan constantes dentro de cada pedido. Si alguna variable presenta varios valores para un mismo pedido, no debemos elegir uno arbitrariamente.

In [6]:
variables_constantes = [
    "Late_delivery_risk",
    "Delivery Status",
    "Customer Segment",
    "Order Region"
]

print("¿Coinciden Order Profit Per Order y Benefit per order en todas las filas?")
print(
    df["Order Profit Per Order"].eq(df["Benefit per order"]).all()
)

print("\nCantidad máxima de valores distintos por pedido:")
display(
    df.groupby("Order Id")[variables_constantes]
    .nunique(dropna=False)
    .max()
    .to_frame("Máximo de valores distintos")
)

print("\n¿Order Item Id identifica cada fila?")
print(f"Valores únicos de Order Item Id: {df['Order Item Id'].nunique():,}")
print(f"Filas de la base: {len(df):,}")

¿Coinciden Order Profit Per Order y Benefit per order en todas las filas?
True

Cantidad máxima de valores distintos por pedido:


,Máximo de valores distintos
Late_delivery_risk,1
Delivery Status,1
Customer Segment,1
Order Region,1



¿Order Item Id identifica cada fila?
Valores únicos de Order Item Id: 180,519
Filas de la base: 180,519


### 1.4. Revisar valores negativos y posibles outliers

Antes de construir una tabla por pedido, revisamos los valores negativos de las variables económicas y detectamos posibles valores atípicos mediante el rango intercuartílico (IQR). Estos controles son exploratorios: una ganancia negativa puede representar una pérdida real y un valor extremo no es necesariamente un error. No eliminamos observaciones automáticamente.

También contrastamos `Order Profit Per Order` con el producto `Order Item Total * Order Item Profit Ratio`. Esta comparación permite evaluar coherencia numérica, pero no demuestra por sí sola que esa multiplicación sea la fórmula oficial de la ganancia.

In [7]:
columnas_economicas = [
    "Order Item Total",
    "Order Item Profit Ratio",
    "Order Profit Per Order",
    "Benefit per order",
    "Sales per customer"
]

resumen_negativos = []
for columna in columnas_economicas:
    valores = pd.to_numeric(df[columna], errors="coerce")
    resumen_negativos.append({
        "variable": columna,
        "negativos": int((valores < 0).sum()),
        "porcentaje_negativo": float((valores < 0).mean() * 100),
        "mínimo": valores.min(),
        "máximo": valores.max()
    })

display(pd.DataFrame(resumen_negativos).round(4))

# Contraste numérico exploratorio entre ganancia registrada y total por ratio.
ganancia_calculada = df["Order Item Total"] * df["Order Item Profit Ratio"]
diferencia = (df["Order Profit Per Order"] - ganancia_calculada).abs()
print("Diferencia absoluta entre Order Profit Per Order y Order Item Total * Order Item Profit Ratio:")
display(diferencia.describe().to_frame("diferencia_absoluta").round(4))
print(f"Filas con diferencia absoluta <= 0,02: {(diferencia <= 0.02).sum():,} ({(diferencia <= 0.02).mean()*100:.1f}%)")
print("La relación no es exacta en todas las filas; no se reemplaza la variable original por esta fórmula.")

# Detección exploratoria de outliers mediante IQR.
resumen_outliers = []
for columna in columnas_economicas:
    valores = pd.to_numeric(df[columna], errors="coerce").dropna()
    q1 = valores.quantile(0.25)
    q3 = valores.quantile(0.75)
    iqr = q3 - q1
    inferior = q1 - 1.5 * iqr
    superior = q3 + 1.5 * iqr
    cantidad = int(((valores < inferior) | (valores > superior)).sum())
    resumen_outliers.append({
        "variable": columna,
        "límite_inferior_IQR": inferior,
        "límite_superior_IQR": superior,
        "posibles_outliers": cantidad,
        "porcentaje": cantidad / len(valores) * 100
    })

display(pd.DataFrame(resumen_outliers).round(4))

,variable,negativos,porcentaje_negativo,mínimo,máximo
0,Order Item Total,0,0.00,7.49,"1,939.99"
1,Order Item Profit Ratio,33784,18.71,-2.75,0.50
2,Order Profit Per Order,33784,18.71,"-4,274.98",911.80
3,Benefit per order,33784,18.71,"-4,274.98",911.80
4,Sales per customer,0,0.00,7.49,"1,939.99"


Diferencia absoluta entre Order Profit Per Order y Order Item Total * Order Item Profit Ratio:


,diferencia_absoluta
count,"180,519.00"
mean,0.22
std,0.41
min,0.00
25%,0.00
50%,0.00
75%,0.33
max,9.30


Filas con diferencia absoluta <= 0,02: 115,474 (64.0%)
La relación no es exacta en todas las filas; no se reemplaza la variable original por esta fórmula.


,variable,límite_inferior_IQR,límite_superior_IQR,posibles_outliers,porcentaje
0,Order Item Total,-110.15,461.93,1943,1.08
1,Order Item Profit Ratio,-0.34,0.78,17300,9.58
2,Order Profit Per Order,-79.70,151.50,18942,10.49
3,Benefit per order,-79.70,151.50,18942,10.49
4,Sales per customer,-110.15,461.93,1943,1.08


**Decisión metodológica:** no se eliminan ganancias negativas ni posibles outliers por el solo hecho de ser negativos o extremos. La variable `Order Profit Per Order` se agrega por `Order Id` como suma de los importes registrados en las filas del pedido, interpretándola como una medida derivada de los registros de artículos. Esta decisión es compatible con que los importes varíen entre filas, pero debe leerse con la limitación documentada: el diccionario no confirma una fórmula de agregación y la relación con `Order Item Total * Order Item Profit Ratio` no es exacta en todas las filas. La columna `margen_ganancia_pedido` se considera una medida derivada exploratoria, no una variable original del dataset.

## Etapa 2 — Preparar las variables necesarias

Trabajamos sobre una copia para conservar intacto el DataFrame original. Normalizamos los espacios sobrantes en `Order Region`, ya que pueden generar categorías duplicadas al agrupar, y convertimos las fechas a tipo fecha para que sean utilizables en análisis posteriores.

No eliminamos filas por tener ganancias negativas ni por aparecer como posibles outliers: primero deben interpretarse según la variable y el contexto. En esta sección se detectan y documentan; no hay evidencia suficiente para corregir o excluir automáticamente esos registros.

In [8]:
df_base_preparada = df.copy()

# Normalizar espacios sobrantes en la región
df_base_preparada["Order Region"] = df_base_preparada["Order Region"].astype("string").str.strip()

# Convertir fechas sin alterar el DataFrame original
columnas_fecha = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

for columna in columnas_fecha:
    if columna in df_base_preparada.columns:
        df_base_preparada[columna] = pd.to_datetime(
            df_base_preparada[columna],
            errors="coerce"
        )

print("Dimensiones después de preparar la copia:", df_base_preparada.shape)
print("¿Order Item Id identifica cada fila?", df_base_preparada["Order Item Id"].is_unique)
print("Cantidad de regiones después de quitar espacios:", df_base_preparada["Order Region"].nunique())
print("\nValores faltantes en variables necesarias:")
display(
    df_base_preparada[
        [
            "Order Id",
            "Order Item Id",
            "Order Profit Per Order",
            "Order Item Total",
            "Late_delivery_risk",
            "Delivery Status",
            "Customer Segment",
            "Order Region"
        ]
    ].isnull().sum()
)

print("\nFechas no interpretadas:")
for columna in columnas_fecha:
    if columna in df_base_preparada.columns:
        print(f"{columna}: {df_base_preparada[columna].isna().sum()}")

Dimensiones después de preparar la copia: (180519, 53)
¿Order Item Id identifica cada fila? True
Cantidad de regiones después de quitar espacios: 23

Valores faltantes en variables necesarias:


,0
Order Id,0
Order Item Id,0
Order Profit Per Order,0
Order Item Total,0
Late_delivery_risk,0
Delivery Status,0
Customer Segment,0
Order Region,0



Fechas no interpretadas:
order date (DateOrders): 0
shipping date (DateOrders): 0


## Etapa 3 — Construir una tabla con un registro por pedido

Agrupamos las filas por `Order Id`. Sumamos los importes de ganancia registrados y `Order Item Total` para construir medidas agregadas por pedido. Antes, comprobamos que el riesgo, el estado de entrega, el segmento y la región sean constantes dentro de cada pedido; si esa condición falla, `first` no resolvería la inconsistencia y habría que investigarla.

La suma de ganancias se presenta como una medida derivada de los registros de la base, con la limitación indicada en la Etapa 1. No se interpreta como una fórmula oficial confirmada por el diccionario. El margen se calcula como ganancia agregada dividida por ventas agregadas y se identifica explícitamente como una medida derivada.

In [9]:
# Comprobación explícita: estas variables deben tener un único valor por pedido.
variables_esperadas_constantes = [
    "Late_delivery_risk", "Delivery Status", "Customer Segment",
    "Order Region", "Order Status", "order date (DateOrders)"
]
consistencia_pedido = df_base_preparada.groupby("Order Id")[variables_esperadas_constantes].nunique(dropna=False)
pedidos_inconsistentes = consistencia_pedido.gt(1).any(axis=1)
print(f"Pedidos con inconsistencias en variables que deberían ser constantes: {pedidos_inconsistentes.sum():,}")

if pedidos_inconsistentes.any():
    display(consistencia_pedido.loc[pedidos_inconsistentes].head(10))
    raise ValueError("Hay pedidos con variables inconsistentes. Revisar antes de usar first() en la agregación.")

df_pedidos = (
    df_base_preparada.groupby("Order Id", as_index=False)
    .agg(
        cantidad_filas=("Order Item Id", "size"),
        unidades_totales=("Order Item Quantity", "sum"),
        ganancia_total_pedido=("Order Profit Per Order", "sum"),
        ventas_totales_pedido=("Order Item Total", "sum"),
        riesgo_entrega_tardia=("Late_delivery_risk", "first"),
        estado_entrega=("Delivery Status", "first"),
        segmento_cliente=("Customer Segment", "first"),
        region=("Order Region", "first"),
        fecha_pedido=("order date (DateOrders)", "first"),
        estado_pedido=("Order Status", "first")
    )
)

df_pedidos["margen_ganancia_pedido"] = (
    df_pedidos["ganancia_total_pedido"]
    / df_pedidos["ventas_totales_pedido"].where(
        df_pedidos["ventas_totales_pedido"] != 0
    )
)

print(f"Pedidos en la tabla final: {len(df_pedidos):,}")
print(f"Pedidos únicos: {df_pedidos['Order Id'].nunique():,}")
display(df_pedidos.head())

Pedidos con inconsistencias en variables que deberían ser constantes: 0
Pedidos en la tabla final: 65,752
Pedidos únicos: 65,752


,Order Id,cantidad_filas,unidades_totales,ganancia_total_pedido,ventas_totales_pedido,riesgo_entrega_tardia,estado_entrega,segmento_cliente,region,fecha_pedido,estado_pedido,margen_ganancia_pedido
0,1,1,1,88.79,239.98,0,Advance shipping,Consumer,Central America,2015-01-01 00:00:00,CLOSED,0.37
1,2,3,7,195.90,529.38,0,Advance shipping,Consumer,South America,2015-01-01 00:21:00,PENDING_PAYMENT,0.37
2,4,4,14,124.09,620.87,1,Late delivery,Home Office,South America,2015-01-01 01:03:00,CLOSED,0.20
3,5,5,10,390.09,987.07,1,Late delivery,Consumer,South America,2015-01-01 01:24:00,COMPLETE,0.40
4,7,3,7,203.93,525.52,1,Late delivery,Consumer,South America,2015-01-01 02:06:00,COMPLETE,0.39


### 3.1. Interpretar las nuevas variables

- `ganancia_total_pedido`: suma de los importes registrados en `Order Profit Per Order` para las filas asociadas al pedido. Es una medida derivada y está sujeta a la limitación metodológica documentada.
- `ventas_totales_pedido`: suma de `Order Item Total` para las filas asociadas al pedido.
- `margen_ganancia_pedido`: cociente entre la ganancia agregada y las ventas agregadas. Es una medida calculada para este análisis, no una columna original.
- `cantidad_filas`: cantidad de registros asociados al pedido.
- `estado_entrega`, `segmento_cliente` y `region`: características conservadas a nivel de pedido después de comprobar su consistencia.

La tabla permite comparar pedidos como unidades de análisis, evitando dar más peso a los pedidos que contienen más filas por el solo hecho de aparecer repetidos en la base original. La validez de la comparación económica depende de la interpretación de la ganancia agregada señalada anteriormente.

## Etapa 4 — Validar la tabla y dejarla lista para el análisis

Antes de entregar la tabla a las siguientes etapas del trabajo, verificamos que cada pedido aparezca una sola vez, que no falten valores en las variables centrales y que las categorías de entrega estén bien identificadas.

Los pedidos con estado `Shipping canceled` se mantienen en `df_pedidos` para no perder información. Creamos por separado `df_pedidos_analisis`, que excluye esos pedidos de la comparación de entregas realizadas, ya que un envío cancelado no es comparable con uno tardío, a tiempo o anticipado. La cantidad excluida se informa explícitamente.

In [10]:
variables_clave = [
    "Order Id",
    "ganancia_total_pedido",
    "ventas_totales_pedido",
    "riesgo_entrega_tardia",
    "estado_entrega",
    "segmento_cliente",
    "region"
]

print("¿Hay un único registro por pedido?")
print(df_pedidos["Order Id"].is_unique)

print("\nValores faltantes en variables clave:")
display(df_pedidos[variables_clave].isnull().sum().to_frame("Valores faltantes"))

print("\nCantidad de pedidos por estado de entrega:")
display(
    df_pedidos["estado_entrega"]
    .value_counts(dropna=False)
    .to_frame("Cantidad de pedidos")
)

cantidad_antes = len(df_pedidos)
df_pedidos_analisis = df_pedidos.loc[
    df_pedidos["estado_entrega"] != "Shipping canceled"
].copy()
cantidad_excluida = cantidad_antes - len(df_pedidos_analisis)

print(f"\nPedidos en la tabla completa: {cantidad_antes:,}")
print(f"Pedidos excluidos por envío cancelado: {cantidad_excluida:,}")
print(f"Pedidos disponibles para comparar entregas: {len(df_pedidos_analisis):,}")

print("\nResumen de la tabla de análisis:")
display(
    df_pedidos_analisis[
        [
            "ganancia_total_pedido",
            "ventas_totales_pedido",
            "margen_ganancia_pedido",
            "riesgo_entrega_tardia"
        ]
    ].describe().T
)

¿Hay un único registro por pedido?
True

Valores faltantes en variables clave:


,Valores faltantes
Order Id,0
ganancia_total_pedido,0
ventas_totales_pedido,0
riesgo_entrega_tardia,0
estado_entrega,0
segmento_cliente,0
region,0



Cantidad de pedidos por estado de entrega:


,Cantidad de pedidos
estado_entrega,
Late delivery,36048
Advance shipping,15127
Shipping on time,11722
Shipping canceled,2855



Pedidos en la tabla completa: 65,752
Pedidos excluidos por envío cancelado: 2,855
Pedidos disponibles para comparar entregas: 62,897

Resumen de la tabla de análisis:


,count,mean,std,min,25%,50%,75%,max
ganancia_total_pedido,"62,897.00",60.52,189.66,"-4,325.85",7.80,72.64,158.77,980.28
ventas_totales_pedido,"62,897.00",503.12,321.29,7.49,244.35,456.87,721.96,"2,768.41"
margen_ganancia_pedido,"62,897.00",0.12,0.37,-2.75,0.05,0.23,0.34,0.50
riesgo_entrega_tardia,"62,897.00",0.57,0.49,0.00,0.00,1.00,1.00,1.00


### Conclusión de la Sección 2

La base original contiene varias filas por pedido, por lo que se preparó una tabla con un registro por `Order Id`. Se revisaron las variables económicas, los valores negativos y los posibles outliers sin eliminarlos automáticamente. También se comprobó la consistencia de las variables que deben representar a cada pedido.

La ganancia agregada y el margen son medidas derivadas de los registros disponibles. El diccionario no confirma una fórmula oficial de agregación, por lo que esta limitación debe tenerse presente en las comparaciones posteriores. Los pedidos con estado `Shipping canceled` permanecen en la tabla completa, pero se excluyen de la tabla específica para comparar entregas realizadas; la exclusión se cuantifica en la etapa de validación.

La tabla `df_pedidos_analisis` queda disponible para los resúmenes estadísticos y gráficos posteriores, siempre que el grupo mantenga explícita la limitación sobre la medida económica utilizada.

## Registro del uso de IA

Se utilizó IA como apoyo para revisar cómo estaban organizados los datos, comprobar si las variables importantes eran consistentes y evaluar las decisiones tomadas para limpiar y preparar la información. También se consultó sobre cómo realizar y verificar algunas transformaciones de los datos utilizando Python y pandas.

**Prompt realizado:** «Ayudame a revisar la preparación de los datos, detectar posibles problemas metodológicos, validar la unidad de análisis y proponer controles».